In [2]:
# ─────────────────────────────────────────────
# Notebook  : nb_gold_transform
# Layer     : Silver → Gold
# Purpose   : Build fact and dimension tables
#             for business analytics
# Author    : Charan Gnanappan
# Date      : 2026-06-24
# ─────────────────────────────────────────────

from pyspark.sql.functions import (
    col, current_timestamp, year, month, 
    dayofmonth, quarter, dayofweek, round
)

# Get Lakehouse path
lh = notebookutils.lakehouse.get("lh_northwind_erp")
LH_PATH = lh.properties["abfsPath"]

print(f"Lakehouse path resolved: {LH_PATH}")
print("Gold notebook ready.")

StatementMeta(, e54b3eb6-3a7f-4b53-8a05-8e9663f1268d, 4, Finished, Available, Finished, False)

Lakehouse path resolved: abfss://42e264c9-ac1b-48f9-975c-2bbaf3322f18@onelake.dfs.fabric.microsoft.com/968d0d78-056b-4db2-a68d-e3d193fb22af
Gold notebook ready.


In [2]:
# Cell 2 — dim_customer

df_customers = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/silver_customers")

dim_customer = df_customers.select(
    col("customerID"),
    col("companyName"),
    col("contactName"),
    col("contactTitle"),
    col("city"),
    col("region"),
    col("country"),
    col("phone")
).withColumn("_gold_processed_at", current_timestamp())

count = dim_customer.count()

dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/dim_customer")

print(f"[dim_customer] {count} rows written successfully")

StatementMeta(, 2b900cfd-29cf-497a-a8ce-1d63e0511119, 4, Finished, Available, Finished, False)

[dim_customer] 91 rows written successfully


In [3]:
# Cell 3 — dim_product

df_products = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/silver_products")
df_categories = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/silver_categories")

dim_product = df_products.join(
    df_categories,
    on="categoryID",
    how="left"
).select(
    col("productID"),
    col("productName"),
    col("categoryID"),
    col("categoryName"),
    col("quantityPerUnit"),
    col("unitPrice"),
    col("discontinued")
).withColumn("_gold_processed_at", current_timestamp())

count = dim_product.count()

dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/dim_product")

print(f"[dim_product] {count} rows written successfully")

StatementMeta(, 2b900cfd-29cf-497a-a8ce-1d63e0511119, 5, Finished, Available, Finished, False)

[dim_product] 77 rows written successfully


In [4]:
# Cell 4 — dim_employee

df_employees = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/silver_employees")

dim_employee = df_employees.select(
    col("employeeID"),
    col("firstName"),
    col("lastName"),
    col("title"),
    col("city"),
    col("region"),
    col("country"),
    col("reportsTo")
).withColumn("_gold_processed_at", current_timestamp())

count = dim_employee.count()

dim_employee.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/dim_employee")

print(f"[dim_employee] {count} rows written successfully")

StatementMeta(, 2b900cfd-29cf-497a-a8ce-1d63e0511119, 6, Finished, Available, Finished, False)

[dim_employee] 9 rows written successfully


In [3]:
# Cell 5 — dim_date (generated, not from Silver)

from pyspark.sql.functions import explode, sequence, to_date, lit
from pyspark.sql.types import DateType

# Generate one row per date between first order and last order
df_date = spark.sql("""
    SELECT explode(sequence(
        to_date('1996-07-04'),
        to_date('1998-05-06'),
        interval 1 day
    )) AS date
""")

dim_date = df_date.select(
    col("date"),
    year(col("date")).alias("year"),
    quarter(col("date")).alias("quarter"),
    month(col("date")).alias("month"),
    dayofmonth(col("date")).alias("day"),
    dayofweek(col("date")).alias("day_of_week")
).withColumn("_gold_processed_at", current_timestamp())

count = dim_date.count()

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/dim_date")

print(f"[dim_date] {count} rows written successfully")

StatementMeta(, e54b3eb6-3a7f-4b53-8a05-8e9663f1268d, 5, Finished, Available, Finished, False)

[dim_date] 672 rows written successfully


In [5]:
# Cell 6 — fact_sales

df_order_details = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/silver_order_details")
df_orders = spark.read.format("delta").load(f"{LH_PATH}/Tables/dbo/silver_orders")

# Join order_details with orders to get customer, employee, date context
df_joined = df_order_details.join(
    df_orders,
    on="orderID",
    how="inner"
)

fact_sales = df_joined.select(
    col("orderID"),
    col("productID"),
    col("customerID"),
    col("employeeID"),
    to_date(col("orderDate")).alias("orderDate"),
    col("quantity"),
    col("unitPrice"),
    col("discount"),
    round(
        col("quantity") * col("unitPrice") * (1 - col("discount")), 
        2
    ).alias("lineTotal"),
    col("freight"),
    col("shipCountry")
).withColumn("_gold_processed_at", current_timestamp())

count = fact_sales.count()

fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{LH_PATH}/Tables/dbo/fact_sales")

print(f"[fact_sales] {count} rows written successfully")

StatementMeta(, e54b3eb6-3a7f-4b53-8a05-8e9663f1268d, 7, Finished, Available, Finished, False)

[fact_sales] 2155 rows written successfully
